In [0]:
%run ../read_params

In [0]:
%run ../utils

In [0]:
%run ./schemas

In [0]:
black_2_white_2_url = f'https://pokeapi.co/api/v2/pokedex/9/'
black_2_white_2_df = api_extraction(black_2_white_2_url, dex_schema)

black_2_white_2_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.black_2_white_2_dex")

In [0]:
black_2_white_2_dex = spark.table(f"{STAGING_DATABASE_PREFIX}.black_2_white_2_dex")

black_2_white_2_dex_df = (
    black_2_white_2_dex
    .withColumn("pokemon_entry", explode(col("pokemon_entries")))
    .select(
        col("pokemon_entry.entry_number").alias("pokedex_number"),
        col("pokemon_entry.pokemon_species.name").alias("pokemon_name"),
        col("pokemon_entry.pokemon_species.url").alias("species_url")
    )
)

black_2_white_2_dex_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{BRONZE_DATABASE_PREFIX}.black_2_white_2_pokedex")

In [0]:
b2w2_expanded_dex = spark.sql(f"""
    WITH b2w2_pokedex AS (
        SELECT 
            np.pokedex_number AS national_pokedex_number
            ,b2w2.*
        FROM    
            bronze.black_2_white_2_pokedex b2w2

        LEFT JOIN bronze.national_pokedex np
        ON b2w2.species_url = np.species_url
    )

    SELECT 
        b2w2.pokedex_number AS b2w2_pokedex_number
        ,s.nat_dex_pokedex_no
        ,s.nat_dex_pokemon_name
        ,s.id AS species_id
        ,s.name AS species_name
        ,s.capture_rate
        ,s.base_happiness
        ,s.growth_rate
        ,s.generation
        ,v.id AS variety_id
        ,v.abilities
        ,v.hidden_ability
        ,v.type_1
        ,v.type_2
        ,v.location_area_encounters
        ,f.id AS form_id
        ,f.form_name
    FROM 
        b2w2_pokedex b2w2

    LEFT JOIN bronze.species s
    ON b2w2.national_pokedex_number = s.nat_dex_pokedex_no

    LEFT JOIN bronze.varieties v
    ON s.id = v.species_id

    LEFT JOIN bronze.forms f
    ON v.id = f.variety_id

    WHERE f.form_name = ""

    ORDER BY    
        b2w2.pokedex_number
""")

b2w2_expanded_dex.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{SILVER_DATABASE_PREFIX}.black_2_white_2_pokedex")

In [0]:
b2w2_locations = b2w2_expanded_dex.select('nat_dex_pokedex_no', 'species_id', 'variety_id', 'location_area_encounters').collect()

b2w2_locations_list = [[row['nat_dex_pokedex_no'], row['species_id'], row['variety_id'], row['location_area_encounters']] for row in b2w2_locations]

for nat_dex_pokedex_no, species_id, variety_id, location_area_encounters_url in b2w2_locations_list:
    print(f'Extracting {nat_dex_pokedex_no}...')

    response = requests.get(location_area_encounters_url)
    encounters_data = response.json()

    base_dict =  {
            'nat_dex_pokedex_no': nat_dex_pokedex_no, 
            'species_id': species_id, 
            'variety_id': variety_id, 
    }
    
    if encounters_data:
        encounters_data_final = {'encounters_data': encounters_data}
    else:
        encounters_data_final = {'encounters_data': []}
    
    encounters_data_dict = base_dict | encounters_data_final
    
    df = spark.createDataFrame([encounters_data_dict], schema=pokemon_locations_schema)

    df.write.format('delta').mode("append").option('mergeSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.pokemon_available_locations")

In [0]:
pokemon_available_locations_df = spark.table(f"{STAGING_DATABASE_PREFIX}.pokemon_available_locations")

encounters_df = (
    pokemon_available_locations_df
    .withColumn("encounter", explode_outer(col("encounters_data")))
    .withColumn('encounter_details', explode_outer(col('encounter.version_details')))
    .withColumn('encounter_details_exploded', explode_outer(col('encounter_details.encounter_details')))
    .select(
        col("nat_dex_pokedex_no"),
        col("species_id"),
        col("variety_id"),
        col("encounter.location_area.name").alias('encounter_location_area'),
        col('encounter_details.version.name').alias('version_name'),
        col('encounter_details.max_chance').alias('encounter_rate'),
        col('encounter_details_exploded.method.name').alias('encounter_method'),
        col('encounter_details_exploded.condition_values').alias('encounter_condition_values'),
        col('encounter_details_exploded.min_level').alias('encounter_min_level'),
        col('encounter_details_exploded.max_level').alias('encounter_max_level')
    )
    .groupby(
        'nat_dex_pokedex_no',
        'species_id',
        'variety_id',
        'encounter_location_area',
        'version_name',
        'encounter_rate',
        'encounter_method',
        'encounter_condition_values'
    ).agg(
        expr('min(encounter_min_level)').alias('encounter_min_level'),
        expr('max(encounter_max_level)').alias('encounter_max_level')
    )
)

encounters_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{BRONZE_DATABASE_PREFIX}.encounters")

In [0]:
encounters_df.filter(col('species_id') == lit(510)).filter(col('version_name') == lit('black-2')).display()

In [0]:
%sql
WITH b2w2_dex_cte AS (
    SELECT 
        b2w2.b2w2_pokedex_number
        ,b2w2.nat_dex_pokedex_no
        ,b2w2.species_id
        ,b2w2.variety_id
        ,b2w2.species_name
        ,ec.evolves_from_variety_id
        ,ec.evolves_from_pokemon_name
        ,ec.evolves_to_variety_id
        ,ec.evolves_to_pokemon_name
    FROM 
        silver.black_2_white_2_pokedex b2w2

    LEFT JOIN silver.evolution_chains ec
    ON b2w2.variety_id = ec.variety_id

    LEFT JOIN bronze.species s
    ON s.id = b2w2.species_id

    WHERE 
        s.generation = 5
)
,encounters_cte AS (
    SELECT
        *
    FROM 
        bronze.encounters e
    WHERE 
        e.version_name = 'black-2'
)
,base_optimal_encounters_cte AS (
    SELECT 
        variety_id
        ,encounter_rate
        ,encounter_location_area
        ,encounter_rate
        ,encounter_method
        ,encounter_condition_values
        ,encounter_min_level
        ,encounter_max_level
        ,1 AS tiebreaker
        ,ROW_NUMBER() OVER (PARTITION BY variety_id ORDER BY encounter_rate DESC) AS rn
    FROM 
        encounters_cte
    WHERE 
        CAST(encounter_condition_values AS VARCHAR(10000)) = '[]'
    AND 
        encounter_method NOT IN ('hidden-grotto', 'grass-spots', 'surf-spots', 'cave-spots', 'bridge-spots')

    UNION 

    SELECT 
        variety_id
        ,CAST(encounter_rate/2 AS INT) AS encounter_rate
        ,encounter_location_area
        ,encounter_rate
        ,encounter_method
        ,encounter_condition_values
        ,encounter_min_level
        ,encounter_max_level
        ,2 AS tiebreaker
        ,ROW_NUMBER() OVER (PARTITION BY variety_id ORDER BY encounter_rate DESC) AS rn
    FROM 
        encounters_cte
    WHERE 
        CAST(encounter_condition_values AS VARCHAR(10000)) = '[]'
    AND 
        encounter_method IN ('hidden-grotto', 'grass-spots', 'surf-spots', 'cave-spots', 'bridge-spots')

    UNION 

    SELECT 
        variety_id
        ,encounter_rate
        ,encounter_location_area
        ,encounter_rate
        ,encounter_method
        ,encounter_condition_values
        ,encounter_min_level
        ,encounter_max_level
        ,3 AS tiebreaker
        ,ROW_NUMBER() OVER (PARTITION BY variety_id ORDER BY encounter_rate DESC) AS rn
    FROM 
        encounters_cte
    WHERE 
        CAST(encounter_condition_values AS VARCHAR(10000)) != '[]'
)
,set_up_optimal_encounters_cte AS (
    SELECT 
        variety_id
        ,encounter_rate
        ,encounter_location_area
        ,encounter_rate
        ,encounter_method
        ,encounter_condition_values
        ,encounter_min_level
        ,encounter_max_level
        ,tiebreaker
        ,ROW_NUMBER() OVER (PARTITION BY variety_id ORDER BY encounter_rate DESC, tiebreaker) AS best_rn
    FROM 
        base_optimal_encounters_cte
    WHERE 
        rn = 1
)
,optimal_encounters_cte AS (
    SELECT 
        variety_id
        ,encounter_rate
        ,encounter_location_area
        ,encounter_rate
        ,encounter_method
        ,encounter_condition_values
        ,encounter_min_level
        ,encounter_max_level
    FROM
        (SELECT * FROM set_up_optimal_encounters_cte WHERE best_rn = 1)
)
,encounterable_cte AS (
    SELECT 
        variety_id
        ,TRUE AS encounterable_in_b2
    FROM 
        encounters_cte
    GROUP BY 
        variety_id
)
,b2_dex_encounterable_cte AS (
    SELECT 
        b2w2.*
        ,COALESCE(e.encounterable_in_b2, FALSE) AS encounterable_in_b2
        ,CASE
            WHEN b2w2.evolves_from_variety_id IS NULL AND b2w2.evolves_to_variety_id IS NULL THEN FALSE
            ELSE TRUE
        END AS can_be_evolved_into
    FROM 
        b2w2_dex_cte b2w2

    LEFT JOIN encounterable_cte e
    ON b2w2.variety_id = e.variety_id
)
,b2_dex_encounterable_final AS (
    SELECT 
        b2_dex.nat_dex_pokedex_no
        ,b2_dex.species_id
        ,b2_dex.variety_id
        ,b2_dex.species_name
        ,CASE 
            WHEN b2_dex.encounterable_in_b2 = FALSE AND b2_dex.can_be_evolved_into = FALSE THEN FALSE
            ELSE TRUE
        END AS obtainable_in_b2
        ,b2_dex.encounterable_in_b2
        ,b2_dex.can_be_evolved_into
        ,CASE 
            WHEN b2_dex.encounterable_in_b2 IS TRUE THEN CONCAT("Encounter ", b2_dex.species_name, " in the wild!")
            WHEN b2_dex.can_be_evolved_into IS FALSE THEN NULL
            WHEN must_evolve_once.encounterable_in_b2 IS TRUE THEN CONCAT("Evolve ", must_evolve_once.species_name, " into ", b2_dex.species_name)
            WHEN must_evolve_once.encounterable_in_b2 IS FALSE AND must_evolve_twice.encounterable_in_b2 IS TRUE THEN CONCAT("Evolve ", must_evolve_twice.species_name, " into ", must_evolve_once.species_name, ", then into ", b2_dex.species_name)
            ELSE "NOT ENCOUNTERABLE IN THE WILD IN BLACK 2!"
        END AS obtainable_how
        ,CASE 
            WHEN b2_dex.encounterable_in_b2 IS TRUE THEN 
                CASE
                    WHEN must_evolve_once.encounterable_in_b2 IS TRUE THEN CONCAT("Evolve ", must_evolve_once.species_name, " into ", b2_dex.species_name)
                    WHEN must_evolve_once.encounterable_in_b2 IS FALSE AND must_evolve_twice.encounterable_in_b2 IS TRUE THEN CONCAT("Evolve ", must_evolve_twice.species_name, " into ", must_evolve_once.species_name, ", then into ", b2_dex.species_name)
                    ELSE NULL
                END
        END AS obtainable_how_alternative
        ,must_evolve_once.encounterable_in_b2 AS pre_evo_encounterable_in_b2
        ,must_evolve_twice.encounterable_in_b2 AS pre_pre_evo_encounterable_in_b2
    FROM 
        b2_dex_encounterable_cte b2_dex

    LEFT JOIN b2_dex_encounterable_cte must_evolve_once
    ON b2_dex.evolves_from_variety_id = must_evolve_once.variety_id
    AND b2_dex.can_be_evolved_into = TRUE

    LEFT JOIN b2_dex_encounterable_cte must_evolve_twice
    ON must_evolve_once.evolves_from_variety_id = must_evolve_twice.variety_id
    AND b2_dex.can_be_evolved_into = TRUE
    AND must_evolve_once.can_be_evolved_into = TRUE
)
SELECT 
    b2.nat_dex_pokedex_no 
    -- ,b2.species_id
    -- ,b2.variety_id
    ,b2.species_name
    ,b2.obtainable_in_b2
    ,b2.obtainable_how
    ,b2.obtainable_how_alternative
    ,e.encounter_location_area
    ,e.encounter_rate
    ,e.encounter_method
    ,e.encounter_condition_values
    ,e.encounter_min_level
    ,e.encounter_max_level
FROM 
    b2_dex_encounterable_final b2

LEFT JOIN optimal_encounters_cte e
ON b2.variety_id = e.variety_id

GROUP BY 
    b2.nat_dex_pokedex_no 
    -- ,b2.species_id
    -- ,b2.variety_id
    ,b2.species_name
    ,b2.obtainable_in_b2
    ,b2.obtainable_how
    ,b2.obtainable_how_alternative
    ,e.encounter_location_area
    ,e.encounter_rate
    ,e.encounter_method
    ,e.encounter_condition_values
    ,e.encounter_min_level
    ,e.encounter_max_level

ORDER BY 
    b2.nat_dex_pokedex_no

In [0]:
T JO%sql
SELECT 
    COALESCE(e.nat_dex_pokedex_no, ec.nat_dex_pokedex_no) AS nat_dex_pokedex_no
    ,COALESCE(e.species_id, ec.species_id) AS species_id
    ,COALESCE(e.variety_id, ec.variety_id) AS variety_id
    ,e.encounter_location_area
    ,CASE 
        WHEN b2w2.b2w2_pokedex_number IS NOT NULL AND e.version_name IS NULL THEN 'black-2'
        ELSE e.version_name
    END AS version_name
    ,e.encounter_rate
    ,e.encounter_method
    ,e.encounter_condition_values
    ,e.encounter_min_level
    ,e.encounter_max_level
FROM 
    silver.black_2_white_2_pokedex b2w2

LEFT JOIN bronze.encounters e
ON e.variety_id = b2w2.variety_id

LEFT JOIN silver.evolution_chains ec
ON b2w2.variety_id = ec.variety_id

LEFT JOIN bronze.species s
ON s.id = b2w2.species_id

WHERE 
    CASE 
        WHEN b2w2.b2w2_pokedex_number IS NOT NULL AND e.version_name IS NULL THEN 'black-2'
        ELSE e.version_name
    END = 'black-2'
AND 
    s.generation = 5

ORDER BY 
    b2w2.b2w2_pokedex_number

In [0]:
%sql
SELECT * FROM silver.evolution_chains